In [ ]:
# !pip install datasets

In [2]:
import re
import pandas as pd
import torch
import torch.nn as nn
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from transformers import AutoTokenizer, BertModel, Trainer, TrainingArguments

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#텍스트 정규화 함수
def normalize(text):
    text = re.sub(r"[^가-힣0-9a-zA-Z\s\.]", ' ', str(text))
    text = re.sub(r"\s+", ' ', text).strip()
    return text

In [4]:
df=pd.read_csv('./ratings_train.txt', sep='\t')

In [5]:
#결측치 제거
df.dropna(inplace=True)
#텍스트 정규화
df['document']=df['document'].map(normalize)

In [ ]:
#길이가 1 이하인 데이터를 필터링
# flag=df['document'].map(lambda x : len(x)) > 1
# flag=df['document'].str.len(x) > 1
df=df.loc[df['document'].str.len() > 1,]
#중복된 데이터를 제거
df.drop_duplicates('document', inplace=True)
df.info()

In [10]:
#인덱스를 초기화 하고 기존의 인덱스는 제거
df.reset_index(drop=True, inplace=True)
df.head(10)

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화 스파이더맨에서 늙어보이기만 했던 커스틴 ...,1
5,5403919,막 걸음마 뗀 3세부터 초등학교 1학년생인 8살용영화. ...별반개도 아까움.,0
6,7797314,원작의 긴장감을 제대로 살려내지못했다.,0
7,9443947,별 반개도 아깝다 욕나온다 이응경 길용우 연기생활이몇년인지..정말 발로해도 그것보단...,0
8,7156791,액션이 없는데도 재미 있는 몇안되는 영화,1
9,5912145,왜케 평점이 낮은건데 꽤 볼만한데.. 헐리우드식 화려함에만 너무 길들여져 있나,1


In [11]:
#label 데이터가 문자라면 숫자로 변환하는 이유는?
1 == '1'

False

In [12]:
df2=df[:5000]

In [13]:
#train,test 데이터 분할
train_df,test_df=train_test_split(
    df2, test_size=0.2, random_state=42, stratify=df2['label']
)

In [15]:
#BERT 모델에서 사용하는 데이터의 타입으로 변환 (parsing)
train_ds=Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds=Dataset.from_pandas(test_df.reset_index(drop=True))

In [16]:
train_ds

Dataset({
    features: ['index', 'id', 'document', 'label'],
    num_rows: 4000
})

In [ ]:
# !pip install tiktoken

In [17]:
#모델을 선택 (토큰화 사전 학습된 모델 모두 같은 모델에서 불러온다.)
MODEL_NAME='beomi/kcbert-base'

#from _pretrained() -> use_fast 매개변수 기본값은 False
    #KoBERT 모델은 sentencepiece 기반 토큰화
#use_fast를 True로 변경하면 (Rust, C 기반)
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

In [27]:
# batch로 묶인 데이터를 토큰화 하는 함수 
def token_fn(batch):
    # batch : Dataset를 이용하여 만들어진 데이터들의 묶음

    result = tokenizer(
        batch['document'], 
        truncation = True, 
        max_length = 128, 
        padding = 'max_length'
    )
    return result

In [28]:
train_tok = train_ds.map(token_fn, batched=True, remove_columns=['id', 'document'])
test_tok = test_ds.map(token_fn, batched=False, remove_columns=['id', 'document'])

Map: 100%|██████████| 1000/1000 [00:04<00:00, 223.73 examples/s]


In [29]:
train_tok
#input_ids : 문장 토큰의 숫자 인덱스 (인코딩 데이터)
#token_type_ids : 문장 구분용 인덱스 (0 : 첫번째 문장, 1 : 두번째 문장, 2 : 버그)
#attention_mask : 실제 토큰과 패딩 토큰 분류 (1 : 실제 토큰, 0 : 패딩 토큰)

Dataset({
    features: ['index', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 4000
})

In [30]:
train_tok['token_type_ids']

Column([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [31]:
#BERT 분류 모델을 선언
class BERTCLF(nn.Module):
    def __init__(self, model_name, num_classes=2, dropout=0.5):
        #model_name : 로드할 모델의 이름
        #num_label : 분류 클래스의 개수
        #dropout : 데이터 소실 비율 (과적합 방지)
        super().__init__()

        #사전에 학습된 모델을 로드
        self.backbone=BertModel.from_pretrained(model_name)

        #출력 차원의 수 로드 (768개 정도)
        hidden=self.backbone.config.hidden_size
        print(hidden)

        #과적합 방지 Dropout
        self.dropout=nn.Dropout(dropout)

        #사전 학습 모델에서 나온 결과를 선형 모델에 대입하기 위해 모델을 생성
        self.fc=nn.Linear(hidden, num_classes)

        #패딩 토큰의 아이디 값을 backbone 설정에 패딩 아이디에 대입 - 확인차 (안정성)
        self.backbone.config.pad_token_id = tokenizer.pad_token_id
        #--- 초기값 설정 완료 ---

    #순전파 함수
    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        #input_ids : 토큰화(인코딩)된 데이터
        #attention_mask : 실제 단어 / 패딩 토큰
        #labels : 학습 시 정답 데이터 (없으면 추론 모드)
        #**kwargs

        #backbone에 데이터 입력
        out=self.backbone(input_ids=input_ids, attention_mask=attention_mask)

        #[CLS] 토큰 벡터를 추출
        #입력의 첫번째 토큰 [CLS] : 문장 전체를 대표하는 의미
        pooled=out.last_hidden_state[:,0]

        #일정 데이터의 소실
        drop_out_data=self.dropout(pooled)

        logits=self.fc(drop_out_data)

        result={'Logits' : logits}

        #labels가 존재한다면 손실을 계산
        if labels is not None:
            loss=nn.CrossEntropyLoss()(logits, labels)
            result['loss']=loss
        return result

In [35]:
#모델 생성
model=BERTCLF(MODEL_NAME)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1758.16it/s]
[transformers] BertModel LOAD REPORT from: beomi/kcbert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


768


In [33]:
from sklearn.metrics import accuracy_score, f1_score

In [40]:
!pip install accelerate>=1.1.0


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [39]:
#TrainingArguments : Trainer가 학습을 할 때 사용히는 각종 설정 값을 지정하는 class
args=TrainingArguments(
    #학습된 모델의 결과를 저장할 경로를 설정
    output_dir='./kobert_from_bertmodel',
    #베치의 크기를 설정 (train batch, vali batch)
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16, 
    #평기, 저장 주기 설정
    eval_strategy='epoch',
    save_strategy='epoch',

    #학습 관련 설정 값
    #1) 반복 횟수
    num_train_epochs=5,
    #2) 옵티마이저 학습률 (최대 보폭)
    learning_rate=5e-5,
    #3) 가중치 감소 개수
    weight_decay=0.01,
    #4) lr읙 값을 올리는 비율
    warmup_ratio=0.1,
    #5) 로그를 출력할 step의 간격
    logging_steps=50,

    #모델의 선택 및 저장 기준
    #학습이 끝났을떄 가장 성능이 좋은 모델을 자동 로드
    load_best_model_at_end=True,
    #최고의 모델을 판단하는 검증 지표
    metric_for_best_model='f1',
    #검증 지표가 높을수록 좋은 모델인가?
    greater_is_better=True,

    #하드웨어 설정
    #cuda 사용 시 16bit 혼합 밀도를 학습할 것인가?
    fp16=torch.cuda.is_available()
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=1.1.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>=1.1.0'`

In [ ]:
#평가 함수 선언
def metrics(eval_pred):
    #eval_pred : 예측값, 실제값
    logits, y = eval_pred
    #logits : [x.xxxx, x.xxxx]
    #logits의 큰 데이터 위치값 변경
    pred=logits.argmax(-1)
    return (
        'accuarcy_score' : accuracy_score(y, pred),
        'f1_score' : f1_score(y, pred)
    )

In [ ]:
#Trainer : 모델이 학습을 자동으로 관리하는 Hugging Face의 고수준 api
trainer=Trainer(
    #모델을 선택
    model=model,
    #학습에서 사용할 설정 값
    args=args,
    #학습에 사용할 데이터
    train_dataset=train_tok,
    #검증에서 사용할 데이터
    eval_dataset=test_tok,
    #토큰화 함수
    tokenizer=tokenizer,
    #평가 시 사용할 검증 지표 함수명
    compute_metrics=metrics
)

In [ ]:
#fine tunning
trainer.train()

In [ ]:
#검증 작업
eval_res=trainer.evaluate()

print(eval_res)